# Import packages

## List of packages :

### Pandas
dataframe handling, allows to get the data

### numpy
data handling, to handle, get or modify values

### datetime
date handling, to allow date conversions


In [ ]:
## import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from folium import Map
from folium.plugins import HeatMap
from datetime import datetime

# Open the csv for reading, get the dataframe

In [ ]:
# open, read the csv
df = pd.read_csv("real_data.csv", sep=";")

## Change columns

In [ ]:
OG_cols = [
    "Date",
    "Service",
    "Departure station",
    "Arrival station",
    "Average journey time",
    "Number of scheduled trains",
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Average delay of late trains at departure",
    "Average delay of all trains at departure",
    "Number of trains delayed at arrival",
    "Average delay of late trains at arrival",
    "Average delay of all trains at arrival",
    "Number of trains delayed > 15min",
    "Average delay of trains > 15min (if competing with flights)",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
    "Pct delay due to external causes",
    "Pct delay due to infrastructure",
    "Pct delay due to traffic management",
    "Pct delay due to rolling stock",
    "Pct delay due to station management and equipment reuse",
    "Pct delay due to passenger handling (crowding, disabled persons, connections)",
]

NEW_Cols = [
    "date",
    "service",
    "gare_depart",
    "gare_arrivee",
    "duree_moyenne",
    "nb_train_prevu",
    "nb_annulation",
    "nb_train_depart_retard",
    "retard_moyen_depart",
    "retard_moyen_tous_trains_depart",
    "nb_train_retard_arrivee",
    "retard_moyen_arrivee",
    "retard_moyen_tous_trains_arrivee",
    "nb_train_retard_sup_15",
    "retard_moyen_trains_retard_sup15",
    "nb_train_retard_sup_30",
    "nb_train_retard_sup_60",
    "prct_cause_externe",
    "prct_cause_infra",
    "prct_cause_gestion_trafic",
    "prct_cause_materiel_roulant",
    "prct_cause_gestion_gare",
    "prct_cause_prise_en_charge_voyageurs",
]

for i in range(len(NEW_Cols)):
    df[NEW_Cols[i]] = df[OG_cols[i]]
    if OG_cols[i] != NEW_Cols[i]:
        df = df.drop(columns=OG_cols[i])

# Data cleaning

## Date

In [ ]:
def normalize_date(s):
    if not isinstance(s, str):
        return np.nan

    s = s.strip()
    # normalize separators
    for i in range(len(s)):
        if not (s[i].isdigit()):
            s = s[0:i] + "-" + s[i + 1 :]
    parts = s.split("-")

    if len(parts) != 2:
        return np.nan

    # detect format:
    # YYYY-(M)M OR (M)M-YYYY
    if len(parts[0]) == 4:
        y, m = parts
    elif len(parts[1]) == 4:
        m, y = parts
    else:
        return np.nan

    # numeric conversion
    if not (y.isdigit() and m.isdigit()):
        return np.nan

    y, m = int(y), int(m)

    # validate month strictly
    if not (1 <= m <= 12):
        return np.nan

    # normalize using calendar
    try:
        base = datetime(y, m, 1)
    except ValueError:
        return np.nan

    return base.strftime("%Y-%b")


df["date"] = df["date"].apply(normalize_date)
df["date"] = pd.to_datetime(df["date"], format="%Y-%b", errors="coerce").dt.strftime(
    "%Y-%m"
)

# then, if a date is nan, check the upper and lower values, and if they are the same, sets this value to this
mask = df["date"].isna() & (df["date"].shift(1) == df["date"].shift(-1))
df.loc[mask, "date"] = df["date"].shift(1)

## Average Travel Time

In [ ]:
def normalize_journey_time(s):
    if isinstance(s, (int, float)):
        return float(s)
    if not isinstance(s, str):
        return np.nan

    s = s.strip()
    s = s.replace(",", ".")
    pos = 0
    for i in s:
        if i == " ":
            break
        pos += 1
    if pos != 0:
        s = s[0:pos]
    else:
        s = "0"
    try:
        float(s)
    except ValueError:
        return np.nan
    return s


df["duree_moyenne"] = df["duree_moyenne"].apply(normalize_journey_time)
df["duree_moyenne"] = pd.to_numeric(df["duree_moyenne"], errors="coerce")

## Comments

In [ ]:
# Ignore these
Comments = [
    "Cancellation comments",
    "Departure delay comments",
    "Arrival delay comments",
]

# remove comments
for i in Comments:
    df = df.drop(columns=i)

## Integer Values

In [ ]:
def normalize_integers(s):
    if not isinstance(s, str) and isinstance(s, (int, float)):
        if pd.isna(s):
            return np.nan
        return float(int(s))
    if not isinstance(s, str):
        return np.nan

    s = s.strip()
    s = s.replace(",", ".")
    if len(s) != 0 and s[0] == "-":
        s = s[1:]
    var = 0
    for i in s:
        if i == " ":
            break
        if not i.isdigit():
            var += 1
    if var > 1:
        return np.nan
    pos = 0
    for i in s:
        if i == " " or i == ".":
            break
        pos += 1
    if pos != 0:
        s = s[0:pos]
    else:
        s = ""
    try:
        int(s)
    except ValueError:
        return np.nan
    return float(int(s))


# Integer values, such as "19", " 19.0 ", etc.
Integer_values = [
    "nb_train_prevu",
    "nb_annulation",
    "nb_train_depart_retard",
    "nb_train_retard_arrivee",
    "nb_train_retard_sup_15",
    "nb_train_retard_sup_30",
    "nb_train_retard_sup_60",
]

# parse integer values as ints
for i in Integer_values:
    df[i] = df[i].apply(normalize_integers)
    df[i] = pd.to_numeric(df[i], errors="coerce")

## Floating point values

In [ ]:
def normalize_numbers(s):
    if isinstance(s, (int, float)):
        return float(s)
    if not isinstance(s, str):
        return np.nan

    if "," in s:
        s = s.replace(",", ".")
    if "%" not in s:
        try:
            float(s)
        except ValueError:
            return np.nan
        return s
    if s[-1] != "%" or ("%" in s[0:-1]):
        return np.nan
    s = s[:-1]
    if len(s) != 0 and s[0] == "-":
        s = s[1:]
    try:
        float(s)
    except ValueError:
        return np.nan
    return s


# Floating point values, such as 0.0, " 185.0 ", etc.
Float_values = [
    "retard_moyen_depart",
    "retard_moyen_tous_trains_depart",
    "retard_moyen_arrivee",
    "retard_moyen_tous_trains_arrivee",
    "retard_moyen_trains_retard_sup15",
    "prct_cause_externe",
    "prct_cause_infra",
    "prct_cause_gestion_trafic",
    "prct_cause_materiel_roulant",
    "prct_cause_gestion_gare",
    "prct_cause_prise_en_charge_voyageurs",
]

# parse floating point values as floats
for i in Float_values:
    df[i] = df[i].apply(normalize_numbers)
    df[i] = pd.to_numeric(df[i], errors="coerce")

## Percentages

In [ ]:
percentages = [
    "prct_cause_externe",
    "prct_cause_infra",
    "prct_cause_gestion_trafic",
    "prct_cause_materiel_roulant",
    "prct_cause_gestion_gare",
    "prct_cause_prise_en_charge_voyageurs",
]

# get the sum of all percentages in each rows
row_sum = df[percentages].sum(axis=1, skipna=True)

# get the number of missing values in each rows
missing_count = df[percentages].isna().sum(axis=1)

# mask of values to fill (there needs to be at least one missing value, and sum must be less than or equal to 100)
mask = (missing_count > 0) & (row_sum <= 100)

# amount to fill per missing value
fill_value = (100 - row_sum) / missing_count

for col in percentages:
    df.loc[mask, col] = df[col].where(df[col].notna(), round(fill_value, 7))

## Fill "very close" values with 0

# re-get sum
row_sum = df[percentages].sum(axis=1, skipna=True)

# find lines with missing values
missing_count = df[percentages].isna().sum(axis=1)

# apply kinder mask
mask = (missing_count > 0) & (row_sum <= 100.1)

for col in percentages:
    df.loc[mask, col] = df[col].where(df[col].notna(), 0)

## Upercase train station's name

In [ ]:
df["gare_depart"] = (
    df["gare_depart"]
    .str.upper()
    .str.replace(
        r"\s+", " ", regex=True
    )  # replace group of multiple spaces into one space
    .str.strip()
    .str.replace(
        r"\bSAINT\b", "ST", regex=True, case=True
    )  # \b is "word boundary" (must be full word) allows for SAINT to become ST, but not ASAINT for example
    .str.replace(r"\bST\.\b", "ST", regex=True, case=True)  # changes ST. into ST (JIC)
)
df["gare_depart"] = df["gare_depart"].mask(
    df["gare_depart"].str.contains(r"\d", na=False)
)

df["gare_arrivee"] = (
    df["gare_arrivee"]
    .str.upper()
    .str.replace(
        r"\s+", " ", regex=True
    )  # replace group of multiple spaces into one space
    .str.strip()
    .str.replace(
        r"\bSAINT\b", "ST", regex=True, case=True
    )  # \b is "word boundary" (must be full word) allows for SAINT to become ST, but not ASAINT for example
    .str.replace(r"\bST\.\b", "ST", regex=True, case=True)  # changes ST. into ST (JIC)
)
df["gare_arrivee"] = df["gare_arrivee"].mask(
    df["gare_arrivee"].str.contains(r"\d", na=False)
)

## Service location cleaning

In [ ]:
cols = ["gare_depart", "gare_arrivee"]

df["service"] = df.groupby(cols)["service"].transform(lambda s: s.ffill().bfill())

# get all cities certified "National"
national_cities = set(
    df.loc[df["service"] == "National", ["gare_depart", "gare_arrivee"]]
    .stack()
    .dropna()
    .unique()
)

# get all missing values and Internationnal values (if they've been mistakenly set to internationnal)
missing = df["service"].isna() | (df["service"] == "International")

# array of lines where the service is National (both cities in the array)
both_national = df["gare_depart"].isin(national_cities) & df["gare_arrivee"].isin(
    national_cities
)

# sets the values for missing values
df.loc[missing, "service"] = np.where(
    both_national[missing], "National", "International"
)

## Remove lines with missing values

The values have been cleaned enough. We are not able to restore gare_depart/gare_arrivee
We are not able to restore the date as well

In [ ]:
df = df.dropna(subset=["date", "gare_depart", "gare_arrivee"])

## Remove duplicate lines to create fullier lines

In [ ]:
df = df.groupby(
    ["date", "service", "gare_depart", "gare_arrivee"],
    sort=False,
    as_index=False,
).first()

## Handle numerical values

In [ ]:
# all num values
Integer_values = Integer_values + ["duree_moyenne"]

# get months
df["date_"] = pd.to_datetime(df["date"], errors="coerce")
df["month"] = df["date_"].dt.month

# try a subset with months
group_month = ["gare_depart", "gare_arrivee", "month"]
group_trajet = ["gare_depart", "gare_arrivee"]

mean_month = df.groupby(group_month)[Integer_values].transform("mean")
mean_route = df.groupby(group_trajet)[Integer_values].transform("mean")
global_mean = df[Integer_values].mean()

df[Integer_values] = df[Integer_values].fillna(round(mean_month, 0))
df[Integer_values] = df[Integer_values].fillna(round(mean_route, 0))
df[Integer_values] = df[Integer_values].fillna(round(global_mean, 0))

mean_month = df.groupby(group_month)[Float_values].transform("mean")
mean_route = df.groupby(group_trajet)[Float_values].transform("mean")
global_mean = df[Float_values].mean()

df[Float_values] = df[Float_values].fillna(mean_month)
df[Float_values] = df[Float_values].fillna(mean_route)
df[Float_values] = df[Float_values].fillna(global_mean)

df = df.drop(columns="date_")
df = df.drop(columns="month")

# Debug print

## dataframe

In [ ]:
print(df)

## infos

In [ ]:
df.info()

## Write in file to check data manually

In [ ]:
df.to_csv("cleaned_dataset.csv", index=False)

# Get the summary statistics

## Get the mean of each number data

In [ ]:
number_values = Integer_values + Float_values

for i in number_values:
    print("Mean of", i.lower(), ":", end=" ")
    print(df[i].mean())

## Get the median of each number value

In [ ]:
for i in number_values:
    print("Median of", i.lower(), ":", end=" ")
    print(df[i].median())

## Get the percentages of departure train stations

In [ ]:
unique_stations = df["gare_depart"].dropna().unique()
total_station = df["gare_depart"].notna().sum()
for i in range(len(unique_stations)):
    station = df["gare_depart"].value_counts().get(unique_stations[i])
    print(
        "Percentage of departure in",
        unique_stations[i],
        ":",
        station / total_station * 100,
        "%",
    )

## Get the percentages of arrival train stations

In [ ]:
unique_stations = df["gare_arrivee"].dropna().unique()
total_station = df["gare_arrivee"].notna().sum()
for i in range(len(unique_stations)):
    station = df["gare_arrivee"].value_counts().get(unique_stations[i])
    print(
        "Percentage of departure in",
        unique_stations[i],
        ":",
        station / total_station * 100,
        "%",
    )

# Visualization of the datas with matplotlib

## Show departure datas with graph (not really readable)

In [ ]:
unique_stations = df["gare_depart"].dropna().unique()

stations = []
for i in range(len(unique_stations)):
    stations.append(df["gare_depart"].value_counts().get(unique_stations[i], 0))

plt.bar(x=unique_stations, height=stations)
plt.xticks(rotation=90)
plt.show()

It is hard to read, so let's filter the datas :

## Show filtered and sorted departure datas to read better

In [ ]:
plt.figure(figsize=(15, 6))

counts = df["gare_depart"].value_counts().head(30)

plt.bar(counts.index, counts.values)
plt.xticks(rotation=90)
plt.show()

We can see that PARIS LYON has the most arrival trains

## Show arrival datas with graph (not really readable)

In [ ]:
unique_stations = df["gare_arrivee"].dropna().unique()

stations = []
for i in range(len(unique_stations)):
    stations.append(df["gare_arrivee"].value_counts().get(unique_stations[i], 0))

plt.bar(x=unique_stations, height=stations)
plt.xticks(rotation=90)
plt.show()

Let's also clean the datas:

## Show filtered and sorted arrival datas to read better

In [ ]:
plt.figure(figsize=(15, 6))

counts = df["gare_arrivee"].value_counts().head(30)

plt.bar(counts.index, counts.values)
plt.xticks(rotation=90)
plt.show()

As the arrivals, PARIS LYON is the train station that have the most departures

## Let's add circle graphs to show the 5 first station with the highest delay time

In [ ]:
delays = (
    df.groupby("gare_depart")["retard_moyen_tous_trains_arrivee"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
)

plt.pie(delays.values, labels=delays.index, autopct="%1.1f%%")
plt.show()

We can see that the train station that have the highest delays is the station that have the most departure : PARIS LYON

## Let's see if it's the same with the gare_arrivees

In [ ]:
delays = (
    df.groupby("gare_arrivee")["retard_moyen_tous_trains_arrivee"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
)

plt.pie(delays.values, labels=delays.index, autopct="%1.1f%%")
plt.show()

## Creating a histogram to show the evolution of the delays

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["Year"] = df["date"].dt.year
retards_par_annee = df["Year"].value_counts().sort_index()
plt.figure(figsize=(10, 6))
plt.bar(retards_par_annee.index, retards_par_annee.values)
plt.xlabel("Year")
plt.ylabel("Number of Delays")
plt.title("Number of Delays per Year")
plt.xticks(retards_par_annee.index)
plt.show()

## Creating a distribution's histogram

In [ ]:
delays = df["retard_moyen_tous_trains_arrivee"].dropna()
delays = delays[delays >= 0]

plt.figure(figsize=(10, 6))
plt.hist(delays, bins=30, edgecolor="black")
plt.xlabel("Average delay (minutes)")
plt.ylabel("Number of routes")
plt.xlim(0, 40)
plt.title("Distribution of arrival delays")
plt.show()

## Adding the correlation heatmap

In [ ]:
num_cols = [
    "retard_moyen_tous_trains_depart",
    "retard_moyen_tous_trains_arrivee",
    "nb_annulation",
    "nb_train_retard_sup_15",
    "retard_moyen_trains_retard_sup15",
    "nb_train_retard_sup_30",
    "nb_train_retard_sup_60",
    "prct_cause_externe",
    "prct_cause_infra",
    "prct_cause_gestion_trafic",
    "prct_cause_materiel_roulant",
]

corr = df[num_cols].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation between delay variables")
plt.tight_layout()
plt.show()

## Showing the uniques stations to create a Heatmap

In [ ]:
print(df["gare_depart"].dropna().unique())

# Creating a heatmap to represent the train stations through France

## Reading the new csv file

In [ ]:
coords = pd.read_csv("train_stations_coordonates.csv")

## Merging the two datasets

In [ ]:
df.columns = df.columns.str.strip()
coords.columns = coords.columns.str.strip()
coords["gare_depart"] = coords["station"]
df = df.merge(coords, on="gare_depart", how="left")
df = df.dropna(subset=["latitude", "longitude"])

## Creating the heatmap's datas

In [ ]:
df["ratio_annulation"] = (
    (df["nb_annulation"] / df["nb_train_prevu"])
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)
heat_data = list(zip(df["latitude"], df["longitude"], df["ratio_annulation"]))
m = Map(location=[46.6, 2.5], zoom_start=6)
HeatMap(heat_data, radius=20, blur=15, min_opacity=0.3).add_to(m)

m

## Saving the heatmap in an html file

In [ ]:
m.save("heatmap.html")